# Task 1：实现 LIF 神经元

目标：模拟**一个神经元**接收恒定电流后，膜电位如何变化、何时发放脉冲。依次运行下方单元格即可。

## 1. 作业公式与 `snn.Leaky`

两者描述同一件事：**输入电流改变膜电位，泄漏使膜电位趋向静息值**。发放和重置前：

- 作业的连续时间方程：$\displaystyle \tau_m\frac{dV(t)}{dt}=-(V(t)-V_{\mathrm{rest}})+RI(t)$。
- `snn.Leaky` 的离散更新：$\displaystyle U_{k+1}=\beta U_k+X_k$。

| 含义 | 作业公式 | `snn.Leaky` | 
|---|---|---|
| 膜电位 | $V$，以 mV 表示实际电位 | $U=V-V_{\mathrm{rest}}$，以静息电位为零点 |
| 泄漏 | $-(V-V_{\mathrm{rest}})$，使电位趋向静息值 | $\beta U$ 保留上一步电位的一部分；差额对应泄漏 |
| 输入 | $RI(t)$ 是输入电流的作用 | $X=\frac{\Delta t}{\tau_m}RI$ 是每步输入增量 |
| 时间尺度 | $\tau_m$ 是膜时间常数 | $\beta=1-\frac{\Delta t}{\tau_m}$ 是 Euler 方法下的保留系数 |

令 $k$ 为时间步编号，$t_k=k\Delta t$，并记 $V_k=V(t_k)$、$I_k=I(t_k)$、$U_k=V_k-V_{\mathrm{rest}}$。使用前向 Euler 近似：

$$\left.\frac{dV}{dt}\right|_{t_k}\approx\frac{V_{k+1}-V_k}{\Delta t}$$

代入连续方程并改用相对电位，得到：

$$U_{k+1}=\left(1-\frac{\Delta t}{\tau_m}\right)U_k+\frac{\Delta t}{\tau_m}RI_k$$

因此上表中的 $\beta$ 和 $X$ 使两种更新对应。一次 `snn.Leaky` 调用对应一个时间步 $\Delta t$；发放、重置和不应期另作处理。默认参数下 $\Delta t=0.1$ ms、$\tau_m=20$ ms、$R=10$ MΩ、$I=2$ nA，得到 $\beta=0.995$、$X=0.1$ mV。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch

## 2. 实现符合题目的 LIF 模块

`LIFNeuron` 封装 snnTorch：用题目的物理参数创建神经元，每次调用传入电流（nA），返回膜电位（mV）和是否发放。模块内部处理张量、阈值重置和不应期；这里用于前向仿真，不用于训练。

In [ ]:
# 作业建议的默认参数：电位 mV、时间 ms、电阻 MΩ、电流 nA
rest, reset, threshold = -65.0, -65.0, -50.0
dt, tau, resistance, current = 0.1, 20.0, 10.0, 2.0
refractory_ms, duration_ms = 2.0, 2000.0
voltage = rest  # 从静息电位开始

In [ ]:
# 对应上面的 U、β 和 X


def relative_voltage(voltage, rest):
    return voltage - rest


def decay_factor(dt, tau):
    return 1 - dt / tau


def input_increment(dt, tau, resistance, current):
    return dt / tau * resistance * current

In [ ]:
def at_threshold(voltage):
    return (voltage >= 0).to(voltage.dtype)


class LIFNeuron(torch.nn.Module):
    def __init__(self, dt, tau, resistance, rest, reset, threshold, refractory_ms):
        super().__init__()
        self.dt, self.tau, self.resistance = dt, tau, resistance
        self.rest, self.reset = rest, reset
        self.refractory_steps = round(refractory_ms / dt)
        self.neuron = snn.Leaky(
            beta=torch.tensor(decay_factor(dt, tau), dtype=torch.float32),
            threshold=relative_voltage(threshold, rest),
            reset_mechanism="zero",
            reset_delay=False,
            spike_grad=at_threshold,
        )
        self.reset_state()

    def reset_state(self):
        self.voltage = relative_voltage(self.reset, self.rest)
        self.remaining = 0

    @torch.no_grad()
    def forward(self, current_na):
        if self.remaining > 0:  # 不应期内保持重置电位
            self.remaining -= 1
            return self.reset, False

        drive = input_increment(self.dt, self.tau, self.resistance, current_na)
        spike, voltage = self.neuron(
            torch.tensor(drive, dtype=torch.float32),
            torch.tensor(self.voltage, dtype=torch.float32),
        )
        spike = bool(spike.item())
        self.voltage = voltage.item()
        if spike:
            self.voltage = relative_voltage(self.reset, self.rest)
            self.remaining = self.refractory_steps
        return self.voltage + self.rest, spike

In [ ]:
def simulate(current_na, duration_ms, dt):
    neuron = LIFNeuron(dt, tau, resistance, rest, reset, threshold, refractory_ms)
    voltages, spikes = [], []
    for _ in range(round(duration_ms / dt)):
        voltage, spike = neuron(current_na)
        voltages.append(voltage)
        spikes.append(spike)
    return np.asarray(voltages), np.asarray(spikes)

## 3. 观察一次仿真

给神经元施加 2 nA 恒定电流，运行 2 秒。图中只展示前 200 ms，红虚线是发放阈值，圆点是脉冲发生的时刻。这里记录的是**每一步结束后的电位**，因此脉冲对应的电位已经重置。

In [ ]:
voltages, spikes = simulate(current_na=current, duration_ms=duration_ms, dt=dt)
time_ms = (np.arange(len(voltages)) + 1) * dt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_ms, voltages, label="Membrane potential")
ax.axhline(threshold, color="red", linestyle="--", label="Threshold")
ax.scatter(time_ms[spikes], voltages[spikes], s=8, label="Spike / reset")
ax.set(xlabel="Time (ms)", ylabel="Voltage (mV)")
ax.set_xlim(0, 200)
ax.legend()
plt.show()
print(
    f"脉冲数：{spikes.sum()}；平均发放率：{spikes.sum() / (len(spikes) * dt / 1000):.1f} Hz"
)

## 4. 用理论检查实现

默认参数下，临界恒定电流为 $(V_{th}-V_{rest})/R=1.5$ nA。低于临界值时电位达不到阈值；恰好等于临界值时电位渐近接近阈值，在有限时间内也不发放。将上面的 2 nA 改为 1 nA、1.5 nA 再运行，观察脉冲是否消失。

这里完成的是 Task 1。不同输入、系统对照实验和滑动发放率属于后续任务。